<a href="https://colab.research.google.com/github/Titantus/Truth-Zero-C/blob/main/T'Z0C_Obloid_Simulation_A_Phase_Space_Atlas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T'Z0C Obloid Simulation: A Phase-Space Atlas

Welcome to the T'Z0C Obloid Simulation Notebook. This environment serves as a dynamic "Phase-Space Atlas," allowing us to explore and quantify the performance of the T'Z0C geometric heat engine under various conditions. Through computational modeling and visualization, we aim to map the operational tolerances and identify optimal configurations for maximal "Sorting Factor."

### Geometric Optimization: Back-Pressure vs. Vortex Coherence

The core of the T'Z0C design lies in its nozzle arrangement. We investigate the critical difference between:

*   **The Back-Pressure Problem (Linear Stacking):** A linear arrangement of nozzles creates a 'Torque Ribbon' with significant back-pressure, leading to localized stalling and capping the maximum torque.
*   **Vortex Coherence (Recursive Spiral):** In contrast, a 3D Recursive Spiral distribution eliminates this stalling by directing nozzle exhaust tangentially and equatorially, feeding the intake of the next nozzle. This induces a macroscopic **Tornadic Vortex**, continuously folding and compressing 'Gray Mode' thermal energy at the equator, creating a coherent momentum engine that aligns perfectly with the oblate geometry. The `go.Cone` traces in the visualization will clearly show this difference, illustrating how the $4.97\%$ gap exhaust is either wasted in competing currents or efficiently channeled into a powerful vortex.

### Material and Medium Influence

Beyond geometry, the operating environment significantly impacts the Obloid's efficiency. This notebook allows us to analyze:

*   **Ambient Atmospheric Air:** The baseline scenario, offering moderate damping and capped torque.
*   **Sealed Noble Gases (Argon, Helium):** Exploring how specific noble gases alter thermal conductivity and molecular impact density, influencing momentum transfer and rotational characteristics.
*   **Partial Vacuum:** Investigating conditions with minimal damping for pure geometric rectification and optimal energy conversion.

The T'Z0C lattice works on any scale, but the viscosity and properties of the fluid interacting with the $4.97\%$ resolution barrier will dictate the actual mechanical output and efficiency. This notebook provides the tools to simulate and understand these critical interactions, guiding the physical prototyping of the Rotary Carbon Harvester (RCH).


In [ ]:
# @title
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

# --- Medium properties (effective coupling to the 4.97% barrier) ---
MEDIUM_PROPERTIES = {
    # name: base_coupling_factor
    "Air (Ambient)": 1.0,        # baseline
    "Argon (sealed)": 1.25,      # heavier, better momentum transfer
    "Helium (sealed)": 0.90,     # light, more slip / less coupling
    "Vacuum (partial)": 1.50     # minimal drag, max rectified flow
}

# --- Frequency response model (FG-200 dial) ---
def frequency_response_factor(
    f_drive_hz: float,
    f_res_hz: float = 1.62e14,
    q_factor: float = 10.0,
    rolloff_power: float = 2.0
) -> float:
    """
    Returns a dimensionless gain factor for how well the drive frequency couples
    into the T'Z0C lattice at the 4.97% resolution barrier.

    - f_drive_hz: actual drive frequency (FG-200 setting, or scaled proxy)
    - f_res_hz:   nominal lattice 'shake' frequency (registry f0)
    - q_factor:   sharpness of the resonance (higher = narrower peak)
    - rolloff_power: how aggressively efficiency falls off away from resonance
    """
    # Normalized detuning
    x = (f_drive_hz - f_res_hz) / (f_res_hz / q_factor)

    # Lorentzian-like envelope with tunable rolloff
    return 1.0 / (1.0 + np.abs(x)**rolloff_power)

def run_tz0c_geometric_sweep(
    medium_name: str = "Air (Ambient)",
    f_drive_hz: float = 1.62e14,
    f_res_hz: float = 1.62e14,
    q_factor: float = 10.0,
    rolloff_power: float = 2.0
):
    """
    Core T'Z0C sweep:
    - geometric sorting factor
    - scaled by medium coupling
    - scaled by frequency response (FG-200 dial)
    """

    # --- Registry / white-paper constants ---
    RESIDUE_GAP = 4.97122
    BASE_SHEAR = 70.53
    PRISM_TARGET = 90.0

    # --- Medium + frequency scaling ---
    medium_factor = MEDIUM_PROPERTIES.get(medium_name, 1.0)
    freq_factor = frequency_response_factor(
        f_drive_hz=f_drive_hz,
        f_res_hz=f_res_hz,
        q_factor=q_factor,
        rolloff_power=rolloff_power
    )
    combined_correction_factor = medium_factor * freq_factor

    results = []
    edge_asymmetries = [1.0, 1.15, 1.3]          # 1:1, 1:1.15, 1:1.30
    dihedral_angles = np.linspace(BASE_SHEAR, PRISM_TARGET, 4)
    bias_shifts = [0.5, 0.7, 0.9]                # 50%, 70%, 90% primary bias

    for edge_bias in edge_asymmetries:
        for angle in dihedral_angles:
            for shift in bias_shifts:
                primary_gap = RESIDUE_GAP * shift
                secondary_gap = (RESIDUE_GAP * (1 - shift)) / 3.0
                bias_delta = primary_gap - secondary_gap

                # geometric kinetic potential (shear -> prism)
                kinetic_potential = np.sin(np.radians(angle))

                sorting_factor = (
                    edge_bias
                    * bias_delta
                    * kinetic_potential
                    * combined_correction_factor
                )

                results.append({
                    "Medium": medium_name,
                    "Edge_Ratio": f"1:{edge_bias:.2f}",
                    "Angle": round(angle, 2),
                    "Primary_Bias_Perc": shift * 100.0,
                    "Sorting_Factor": round(sorting_factor, 5),
                    "Medium_Factor": medium_factor,
                    "Freq_Factor": round(freq_factor, 5),
                    "f_drive_hz": f_drive_hz
                })

    return results

def run_multi_medium_frequency_sweep(
    media=None,
    drive_frequencies_hz=None
):
    if media is None:
        media = list(MEDIUM_PROPERTIES.keys())
    if drive_frequencies_hz is None:
        # e.g. sweep around f0 on a log-ish scale or simple multiples
        f0 = 1.62e14
        drive_frequencies_hz = [
            0.5 * f0,
            0.75 * f0,
            1.0 * f0,
            1.25 * f0,
            1.5 * f0
        ]

    all_rows = []
    for medium_name in media:
        for f_drive in drive_frequencies_hz:
            all_rows.extend(
                run_tz0c_geometric_sweep(
                    medium_name=medium_name,
                    f_drive_hz=f_drive,
                    f_res_hz=1.62e14,
                    q_factor=10.0,
                    rolloff_power=2.0
                )
            )

    return pd.DataFrame(all_rows)

# Execute the multi-medium, multi-frequency sweep
df = run_multi_medium_frequency_sweep()

# --- Multi-panel Heatmap Display ---
# Filter for a specific frequency for plotting, e.g., the resonant frequency
resonant_freq_df = df[df['f_drive_hz'] == 1.62e14].copy() # Use .copy() to avoid SettingWithCopyWarning

media_list = resonant_freq_df['Medium'].unique()
edge_list = resonant_freq_df['Edge_Ratio'].unique()

fig, axes = plt.subplots(len(media_list), len(edge_list), figsize=(5*len(edge_list), 4*len(media_list)), sharex=True, sharey=True)

if len(media_list) == 1 and len(edge_list) == 1:
    axes = np.array([[axes]])
elif len(media_list) == 1:
    axes = axes[np.newaxis, :]
elif len(edge_list) == 1:
    axes = axes[:, np.newaxis]

for i, medium in enumerate(media_list):
    for j, edge in enumerate(edge_list):
        subset = resonant_freq_df[(resonant_freq_df['Medium'] == medium) & (resonant_freq_df['Edge_Ratio'] == edge)]
        pivot = subset.pivot_table(index="Primary_Bias_Perc", columns="Angle", values="Sorting_Factor")
        sns.heatmap(pivot, ax=axes[i, j], cmap='viridis', annot=True, fmt=".2f", cbar=(j==len(edge_list)-1),
                    cbar_kws={'label': 'Sorting Factor'})
        axes[i, j].set_title(f"{medium} | Edge {edge}", fontsize=10)
        axes[i, j].set_xlabel("Angle")
        axes[i, j].set_ylabel("Primary Bias %")

plt.suptitle(f"Sorting Factor Heatmaps for All Media and Edge Ratios at f_drive_hz=1.62e14", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# --- Golden Zone Identification ---
golden_zone = resonant_freq_df[(resonant_freq_df['Angle'] == 90.0) & (resonant_freq_df['Primary_Bias_Perc'] == 90.0)]
if not golden_zone.empty:
    print("--- Golden Zone (90° / 90% Bias) Coordinates (at f_drive_hz=1.62e14) ---")
    for _, row in golden_zone.iterrows():
        print(f"Medium: {row['Medium']}, Edge Ratio: {row['Edge_Ratio']}, Sorting Factor: {row['Sorting_Factor']}")
else:
    print("No exact golden zone found in current sweep (at f_drive_hz=1.62e14).")


In [ ]:
# @title 3D T'Z0C Obloid Vortex Visualization (Plotly)
import numpy as np
import plotly.graph_objects as go

# ==========================================
# 1. T'Z0C Obloid Geometry Parameters
# ==========================================
eq_radius = 1.0  # Equatorial Radius (a)
pol_radius = 0.65 # Polar Radius (c) - creating the Oblate Spheroid constraint

# Generate the Oblate Shell Mesh (The "Housing")
u = np.linspace(0, 2 * np.pi, 60)
v = np.linspace(0, np.pi, 60)
u_grid, v_grid = np.meshgrid(u, v)

X_shell = eq_radius * np.sin(v_grid) * np.cos(u_grid)
Y_shell = eq_radius * np.sin(v_grid) * np.sin(u_grid)
Z_shell = pol_radius * np.cos(v_grid)

# ==========================================
# 2. Linear Stack (The "Stall" Configuration)
# ==========================================
N_linear = 20 # 20 N-spoke base
theta_lin = np.linspace(0, 2 * np.pi, N_linear, endpoint=False)

# Placed linearly along the equator
X_lin = eq_radius * np.cos(theta_lin)
Y_lin = eq_radius * np.sin(theta_lin)
Z_lin = np.zeros(N_linear)

# Vectors (representing the 5.60 Sorting Factor thrust)
# Tangential thrust creates localized ring stalling
U_lin = -Y_lin
V_lin = X_lin
W_lin = np.zeros(N_linear)

# ==========================================
# 3. Recursive Spiral (The "Coherent Vortex" Configuration)
# ==========================================
N_spiral = 60 # Higher density for full internal climate
golden_ratio = (1 + 5**0.5) / 2
indices = np.arange(0, N_spiral, dtype=float) + 0.5

# Map Fibonacci spiral to the oblate spheroid geometry
phi_spir = np.arccos(1 - 2 * indices / N_spiral)
theta_spir = 2 * np.pi * indices / golden_ratio

X_spir = eq_radius * np.sin(phi_spir) * np.cos(theta_spir)
Y_spir = eq_radius * np.sin(phi_spir) * np.sin(theta_spir)
Z_spir = pol_radius * np.cos(phi_spir)

# Vectors (Spiral exhaust)
# Pointing tangentially BUT folding equatorially to induce the vortex
U_spir = -Y_spir
V_spir = X_spir
# Z-vector gently pushes flow toward the equator to compress the Gray Mode
W_spir = -0.3 * Z_spir

# ==========================================
# 4. Plotly Figure Assembly
# ==========================================
fig = go.Figure()

# Add the Translucent Obloid Shell
fig.add_trace(go.Surface(
    x=X_shell, y=Y_shell, z=Z_shell,
    opacity=0.15, colorscale='Blues', showscale=False,
    name="Obloid Housing", hoverinfo="skip"
))

# Add Linear Stack Vectors (Red = High Friction/Stall)
fig.add_trace(go.Cone(
    x=X_lin, y=Y_lin, z=Z_lin, u=U_lin, v=V_lin, w=W_lin,
    sizemode="absolute", sizeref=0.2, anchor="tail",
    colorscale='Reds', showscale=False, name="Linear Stack (Stall)"
))

# Add Recursive Spiral Vectors (Green = High Coherence/Vortex)
fig.add_trace(go.Cone(
    x=X_spir, y=Y_spir, z=Z_spir, u=U_spir, v=V_spir, w=W_spir,
    sizemode="absolute", sizeref=0.15, anchor="tail",
    colorscale='Greens', showscale=False, name="Recursive Spiral (Vortex)",
    visible=False # Hidden by default
))

# ==========================================
# 5. UI Controls & Formatting
# ==========================================
fig.update_layout(
    title="T'Z0C Obloid Nozzle Arrangement Simulator",
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode='data'
    ),
    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=0.5, y=0.05,
        xanchor="center",
        buttons=list([
            dict(
                args=[{"visible": [True, True, False]}],
                label="Linear Stack (Back-Pressure)",
                method="update"
            ),
            dict(
                args=[{"visible": [True, False, True]}],
                label="Recursive Spiral (Vortex Coherence)",
                method="update"
            )
        ])
    )],
    paper_bgcolor="black",
    font=dict(color="white"),
    template="plotly_dark",
    height=700
)

# Render
fig.show()
